# **Pre-processing all PDFs**

Parsing, chunking, embedding, and indexing the full `data_analysis_final/all_pdfs` corpus with resumable per-paper checkpoints.

## Workflow

This notebook processes every PDF in `data_analysis_final/all_pdfs` one at a time. After each successful paper, the Chroma index is persisted and a JSON manifest is updated so reruns can skip completed files.

In [21]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

from langchain_core.documents import Document

workspace_root = Path(r"C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow")
#pdf_source_dir = workspace_root / "data_analysis_final" / "all_pdfs"
pdf_source_dir = Path(r"D:\3. Research Main Room\Publications\1 Current Projects\Data Science and Tourism Research - Conceptual Study\DATA ANALYSIS\Data_files\all_pdfs")
output_root = workspace_root / "data_analysis_final" / "vector_database" / "all_pdfs_index"
chroma_directory = output_root / "chroma"
state_directory = output_root / "state"

output_root.mkdir(parents=True, exist_ok=True)
chroma_directory.mkdir(parents=True, exist_ok=True)
state_directory.mkdir(parents=True, exist_ok=True)

processed_state_path = state_directory / "processed_state.json"


def load_json(path, default):
    if not path.exists():
        return default

    try:
        with path.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except Exception as exc:
        print(f"Warning: could not read {path.name}: {exc}")
        return default


def save_json(path, payload):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    temp_path.replace(path)


processed_state = load_json(processed_state_path, {"completed": {}, "failed": {}})

print(f"PDF source folder: {pdf_source_dir}")
print(f"Output root: {output_root}")
print(f"Completed papers in manifest: {len(processed_state['completed'])}")

PDF source folder: D:\3. Research Main Room\Publications\1 Current Projects\Data Science and Tourism Research - Conceptual Study\DATA ANALYSIS\Data_files\all_pdfs
Output root: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\data_analysis_final\vector_database\all_pdfs_index
Completed papers in manifest: 787


In [22]:
def now_iso():
    return datetime.now(timezone.utc).isoformat()


def list_pdf_files(folder):
    return sorted(
        path for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() == ".pdf"
    )


pdf_files = list_pdf_files(pdf_source_dir)
pending_pdf_files = [pdf_path for pdf_path in pdf_files if pdf_path.stem not in processed_state["completed"]]

print(f"PDFs discovered: {len(pdf_files)}")
print(f"Pending after manifest check: {len(pending_pdf_files)}")

PDFs discovered: 788
Pending after manifest check: 1


### Docling setup

In [23]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions, TableFormerMode

pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.do_code_enrichment = False
pipeline_options.do_formula_enrichment = False
pipeline_options.do_picture_description = False
pipeline_options.do_table_structure = True
pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
    mode=TableFormerMode.ACCURATE,
)
pipeline_options.enable_remote_services = False
pipeline_options.allow_external_plugins = True

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options,
        )
    }
)

print("Docling converter ready.")

Docling converter ready.


### Chunk normalization

In [24]:
def normalize_text(text):
    text = text or ""

    replacements = {
        "/uniFB01": "fi",
        "/uniFB02": "fl",
        "/C15": "",
        "þ": "+",
    }

    for old_value, new_value in replacements.items():
        text = text.replace(old_value, new_value)

    text = re.sub(r"\b(\w+)\s+(fi|fl)\s+(\w+)\b", r"\1\2\3", text)
    text = re.sub(r"\b(\w+)\s+'\s+(\w+)\b", r"\1'\2", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def convert_docling_chunks(doc_splits):
    clean_documents = []

    for chunk_index, doc in enumerate(doc_splits):
        dl_meta = doc.metadata.get("dl_meta", {}) or {}
        headings = dl_meta.get("headings", []) or []

        current_heading = normalize_text(str(headings[0])) if headings else ""
        if re.search(r"(?i)^(references|bibliography)$", current_heading):
            continue

        origin = dl_meta.get("origin", {}) or {}
        doc_items = dl_meta.get("doc_items", []) or []

        page_no = None
        if doc_items:
            prov = doc_items[0].get("prov", []) or []
            if prov:
                page_no = prov[0].get("page_no")

        filename = origin.get("filename", "unknown.pdf")
        paper_id = Path(filename).stem if filename != "unknown.pdf" else "unknown"
        cleaned_text = normalize_text(doc.page_content)

        clean_documents.append(
            Document(
                page_content=cleaned_text,
                metadata={
                    "paper_id": paper_id,
                    "filename": filename,
                    "section_heading": current_heading,
                    "page_no": page_no,
                    "modality": "text",
                    "chunk_id": f"{paper_id}_chunk_{chunk_index}",
                },
            )
        )

    return clean_documents

### Vector store

In [25]:
from langchain_community.vectorstores import Chroma
from langchain_docling.loader import DoclingLoader
from langchain_openai import OpenAIEmbeddings

collection_name = "all_pdfs_text_chunks_v1"
embedding_model = "text-embedding-3-small"

vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=OpenAIEmbeddings(model=embedding_model),
    persist_directory=str(chroma_directory),
)


def update_manifest(paper_id, pdf_path, chunk_count, status):
    processed_state["completed"][paper_id] = {
        "status": status,
        "chunk_count": chunk_count,
        "source_path": str(pdf_path),
        "updated_at": now_iso(),
    }
    processed_state["failed"].pop(paper_id, None)
    save_json(processed_state_path, processed_state)


def record_failure(paper_id, pdf_path, error_message):
    processed_state["failed"][paper_id] = {
        "source_path": str(pdf_path),
        "error": error_message,
        "updated_at": now_iso(),
    }
    save_json(processed_state_path, processed_state)


def get_existing_chunk_count(paper_id):
    results = vectorstore.get(where={"paper_id": paper_id})
    return len(results.get("ids", []))


print(f"Chroma collection: {collection_name}")
print(f"Embedding model: {embedding_model}")
print(f"Chroma directory: {chroma_directory}")

Chroma collection: all_pdfs_text_chunks_v1
Embedding model: text-embedding-3-small
Chroma directory: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\data_analysis_final\vector_database\all_pdfs_index\chroma


### Run preprocessing

In [26]:
from tqdm.auto import tqdm
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()

max_papers_to_process = None  # set to an integer when testing
processed_this_run = 0
total_completed = len(processed_state["completed"])
print(f"Already processed in manifest: {total_completed}")

with tqdm(total=len(pdf_files), initial=total_completed, desc="Preprocessing PDFs", unit="pdf") as progress_bar:
    for paper_number, pdf_path in enumerate(pdf_files, start=1):
        paper_id = pdf_path.stem

        if max_papers_to_process is not None and processed_this_run >= max_papers_to_process:
            break

        progress_bar.set_postfix_str(f"{paper_id} | checking")

        if paper_id in processed_state["completed"]:
            progress_bar.set_postfix_str(f"{paper_id} | skip")
            continue

        existing_chunk_count = get_existing_chunk_count(paper_id)
        if existing_chunk_count > 0:
            update_manifest(paper_id, pdf_path, existing_chunk_count, status="existing")
            progress_bar.set_postfix_str(f"{paper_id} | indexed")
            continue

        try:
            progress_bar.set_postfix_str(f"{paper_id} | parsing")
            loader = DoclingLoader(file_path=str(pdf_path), converter=doc_converter)
            docs = loader.load()

            progress_bar.set_postfix_str(f"{paper_id} | chunking")
            clean_docs = convert_docling_chunks(docs)
            if not clean_docs:
                update_manifest(paper_id, pdf_path, 0, status="empty")
                progress_bar.set_postfix_str(f"{paper_id} | empty")
                continue

            progress_bar.set_postfix_str(f"{paper_id} | embedding")
            chunk_ids = [doc.metadata["chunk_id"] for doc in clean_docs]
            vectorstore.add_documents(clean_docs, ids=chunk_ids)

            progress_bar.set_postfix_str(f"{paper_id} | indexing")
            vectorstore.persist()

            update_manifest(paper_id, pdf_path, len(clean_docs), status="indexed")
            processed_this_run += 1
            progress_bar.update(1)

        except Exception as exc:
            record_failure(paper_id, pdf_path, str(exc))
            progress_bar.set_postfix_str(f"{paper_id} | failed")
            progress_bar.update(1)

Already processed in manifest: 787


Preprocessing PDFs: 100%|#########9| 787/788 [00:00<?, ?pdf/s]

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

C:\Users\sahil\AppData\Local\Temp\ipykernel_11168\3783498034.py:47: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [27]:
completed_count = len(processed_state["completed"])
failed_count = len(processed_state["failed"])
remaining_count = len([pdf_path for pdf_path in pdf_files if pdf_path.stem not in processed_state["completed"]])

print("Run summary")
print(f"  completed: {completed_count}")
print(f"  processed this run: {processed_this_run}")
print(f"  failed: {failed_count}")
print(f"  remaining: {remaining_count}")
print(f"  manifest: {processed_state_path}")
print(f"  Chroma directory: {chroma_directory}")

Run summary
  completed: 788
  processed this run: 1
  failed: 0
  remaining: 0
  manifest: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\data_analysis_final\vector_database\all_pdfs_index\state\processed_state.json
  Chroma directory: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow\data_analysis_final\vector_database\all_pdfs_index\chroma
